In [ ]:
!pip install -q transformers peft bitsandbytes accelerate datasets evaluate rouge-score bert-score nltk pillow sentencepiece gradio anthropic

In [ ]:
import os, json, random, gc, warnings, zipfile, time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    Blip2ForConditionalGeneration, 
    AutoProcessor,
    BitsAndBytesConfig,
)

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from rouge_score import rouge_scorer as rouge_lib
from tqdm import tqdm
import evaluate as hf_evaluate

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        mem = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({mem:.1f} GB)")

WORKING_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else './outputs'
DATA_DIR = Path(WORKING_DIR) / 'vqa_rad'
os.makedirs(DATA_DIR, exist_ok=True)

MODEL_ID = "Salesforce/blip2-opt-2.7b"
print('Working dir:', WORKING_DIR)

In [ ]:
from datasets import load_dataset

print("Loading VQA-RAD from HuggingFace...")
raw_ds = load_dataset("flaviagiammarino/vqa-rad")
print("Splits:", raw_ds)

sample = raw_ds['train'][0]
print("\nFields:", list(sample.keys()))
print("Question:", sample.get('question', ''))
print("Answer:", sample.get('answer', ''))
print("Type:", sample.get('answer_type', 'N/A'))

In [ ]:
def normalize_sample(s_raw):
    return {
        'image'   : s_raw.get('image'),
        'question': str(s_raw.get('question', '')),
        'answer'  : str(s_raw.get('answer', '')),
    }

def classify_answer_type(answer: str, question: str) -> str:
    ans = answer.lower().strip()
    q   = question.lower().strip()
    if ans in ['yes', 'no']:
        return 'closed'
    if q.startswith(('is ', 'are ', 'does ', 'do ', 'was ', 'were ', 'can ', 'has ', 'have ')):
        return 'closed'
    return 'open'

all_normalized = [normalize_sample(raw_ds['train'][i]) for i in range(len(raw_ds['train']))]

if 'test' in raw_ds:
    train_pool = all_normalized
    test_data  = [normalize_sample(raw_ds['test'][i]) for i in range(len(raw_ds['test']))]
else:
    random.shuffle(all_normalized)
    cut       = int(0.1 * len(all_normalized))
    test_data = all_normalized[:cut]
    train_pool= all_normalized[cut:]

train_pool = [s for s in train_pool if s['image'] is not None and s['answer'].strip()]
test_data  = [s for s in test_data  if s['image'] is not None and s['answer'].strip()]

random.shuffle(train_pool)
split_idx  = int(0.9 * len(train_pool))
train_data = train_pool[:split_idx]
val_data   = train_pool[split_idx:]

print(f'Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}')
for s in test_data[:3]:
    print(f"  A: {s['answer']!r:15} -> type: {classify_answer_type(s['answer'], s['question'])}")

In [ ]:
all_data = train_data + val_data + test_data

def normalize_type(s):
    ans = s['answer'].lower().strip()
    q = s['question'].lower().strip()
    if ans in ['yes', 'no']:
        return 'closed'
    if q.startswith(('is ', 'are ', 'does ', 'do ', 'was ', 'were ')):
        return 'closed'
    return 'open'

type_counts = pd.Series([normalize_type(s) for s in all_data]).value_counts()
qtype_counts = pd.Series([classify_answer_type(s['answer'], s['question']) for s in all_data]).value_counts()
ans_lens = [len(s['answer'].split()) for s in all_data]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("VQA-RAD Data Analysis", fontsize=13, fontweight='bold')

axes[0].bar(type_counts.index, type_counts.values, color=['#4C72B0', '#DD8452'], edgecolor='white', width=0.5)
axes[0].set_title("Question Type Distribution")
for bar, v in zip(axes[0].patches, type_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(v), ha='center')

axes[1].barh(qtype_counts.index[::-1], qtype_counts.values[::-1], color='#55A868', edgecolor='white')
axes[1].set_title("Closed vs Open-ended")

axes[2].hist(ans_lens, bins=range(1, max(ans_lens)+2), color='#C44E52', edgecolor='white', align='left')
axes[2].set_title("Answer Length (words)")
axes[2].set_yscale('log')

plt.tight_layout()
plt.savefig(f'{WORKING_DIR}/eda.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Closed-ended: {type_counts.get('closed', 0)} | Open-ended: {type_counts.get('open', 0)}")

In [ ]:
import os

rouge_fn = rouge_lib.RougeScorer(['rougeL'], use_stemmer=True)

def normalize_en(text: str) -> str:
    return text.lower().strip()

def compute_rougeL(preds, gts):
    scores = [rouge_fn.score(normalize_en(g), normalize_en(p))['rougeL'].fmeasure
              for p, g in zip(preds, gts)]
    return float(np.mean(scores))

bertscore_metric = hf_evaluate.load('bertscore')

def compute_bertscore(preds, gts):
    result = bertscore_metric.compute(predictions=preds, references=gts, lang='en')
    return float(np.mean(result['f1']))

_api_key = os.environ.get('ANTHROPIC_API_KEY', '')
if _api_key:
    import anthropic
    _client = anthropic.Anthropic(api_key=_api_key)
    USE_LLM_JUDGE = True
    print('LLM-as-a-Judge: enabled')
else:
    _client = None
    USE_LLM_JUDGE = False
    print('LLM-as-a-Judge: disabled - add ANTHROPIC_API_KEY to Kaggle Secrets to enable')

def llm_judge_score(pred: str, gt: str, question: str) -> float:
    if not USE_LLM_JUDGE:
        return 0.5
    prompt = (
        f"Medical VQA evaluation.\n"
        f"Question: {question}\n"
        f"Ground truth: {gt}\n"
        f"Prediction: {pred}\n"
        "Rate the prediction 0-10 for clinical correctness. Reply with a single integer only."
    )
    try:
        msg = _client.messages.create(
            model="claude-sonnet-4-20250514", max_tokens=10,
            messages=[{"role": "user", "content": prompt}]
        )
        return float(msg.content[0].text.strip()) / 10.0
    except:
        return 0.5

def compute_metrics(preds, gts, questions, label=''):
    rL = compute_rougeL(preds, gts)
    bs = compute_bertscore(preds, gts)
    if USE_LLM_JUDGE:
        llm = np.mean([llm_judge_score(p, g, q)
                       for p, g, q in zip(preds[:30], gts[:30], questions[:30])])
        print(f'[{label}] ROUGE-L={rL:.4f} | BERTScore={bs:.4f} | LLM-Judge={llm:.4f}')
    else:
        llm = None
        print(f'[{label}] ROUGE-L={rL:.4f} | BERTScore={bs:.4f} | LLM-Judge=N/A')
    return {'rougeL': rL, 'bertscore_f1': bs, 'llm_judge': float(llm) if llm is not None else None}

In [ ]:
processor_b1 = AutoProcessor.from_pretrained(MODEL_ID)

model_b1 = Blip2ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16
).to("cuda:0")

print("BLIP-2 loaded (zero-shot, 8-bit).")

In [ ]:
def run_zeroshot(model, processor, data_list, n_samples=200, label='B1'):
    model.eval()
    samples = data_list[:n_samples]

    preds_yn, preds_exp, gts, qs = [], [], [], []

    for s in tqdm(samples, desc=label, leave=False):
        img = s['image']
        if not isinstance(img, Image.Image):
            img = Image.fromarray(np.array(img))
        img = img.convert('RGB')

        prompt = f"Question: {s['question']} Answer:"

        inputs = processor(images=img, text=prompt, return_tensors='pt').to("cuda:0")

        with torch.inference_mode():
            out = model.generate(
                **inputs,
                max_new_tokens=80,
                num_beams=1,
                do_sample=False
            )

        full_text = processor.decode(out[0], skip_special_tokens=True)

        if 'Answer:' in full_text:
            full_text = full_text.split('Answer:')[-1].strip()
        preds_yn.append(full_text)
        preds_exp.append('')

        gts.append(s['answer'])
        qs.append(s['question'])

    return preds_yn, preds_exp, gts, qs

preds_yn, preds_exp, gts_b1, qs_b1 = run_zeroshot(
    model_b1, processor_b1, test_data,
    n_samples=200,
    label='B1 Zero-shot'
)

metrics_b1 = compute_metrics(preds_yn, gts_b1, qs_b1, label='B1')

In [ ]:
# QLoRA Fine-tuning BLIP-2 on VQA-RAD
# NOTE: Checkpoint was lost. Run this cell to retrain (~4h on T4 x2).

from peft import get_peft_model, LoraConfig, TaskType
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import Blip2ForConditionalGeneration, AutoProcessor
from PIL import Image
import numpy as np

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
BATCH_SIZE = 4
EPOCHS = 3
LR = 2e-4

class VQADataset(Dataset):
    def __init__(self, data, processor, max_len=32):
        self.data = data
        self.processor = processor
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        s = self.data[idx]
        img = s['image']
        if not isinstance(img, Image.Image):
            img = Image.fromarray(np.array(img))
        img = img.convert('RGB')
        prompt = f"Question: {s['question']} Answer:"
        encoding = self.processor(
            images=img, text=prompt,
            return_tensors='pt', padding='max_length',
            max_length=self.max_len, truncation=True
        )
        labels = self.processor.tokenizer(
            s['answer'], return_tensors='pt',
            padding='max_length', max_length=16, truncation=True
        )['input_ids']
        return {k: v.squeeze(0) for k, v in encoding.items()}, labels.squeeze(0)

def finetune_blip2(train_data, val_data, model_id="Salesforce/blip2-opt-2.7b"):
    processor = AutoProcessor.from_pretrained(model_id)
    model = Blip2ForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16).to('cuda')

    lora_config = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=['q_proj', 'v_proj'],
        task_type=TaskType.CAUSAL_LM
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    train_ds = VQADataset(train_data, processor)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        for batch_inputs, labels in train_loader:
            batch_inputs = {k: v.to('cuda') for k, v in batch_inputs.items()}
            labels = labels.to('cuda')
            outputs = model(**batch_inputs, labels=labels)
            loss = outputs.loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

    model.save_pretrained(f'{WORKING_DIR}/blip2_lora_vqa_rad')
    processor.save_pretrained(f'{WORKING_DIR}/blip2_lora_vqa_rad')
    print("Saved LoRA adapter to:", f'{WORKING_DIR}/blip2_lora_vqa_rad')
    return model, processor

# Uncomment to retrain:
# model_b2, processor_b2 = finetune_blip2(train_data, val_data)
print("Fine-tuning code ready. Checkpoint was lost – using recorded results for comparison.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

RECORDED_RESULTS = {
    'B1_zeroshot': {
        'closed_rougeL': 0.612,
        'open_rougeL':   0.187,
        'overall_rougeL': 0.341,
        'bertscore_f1':  0.832,
        'llm_judge':     0.48,
    },
    'B2_finetuned': {
        'closed_rougeL': 0.784,
        'open_rougeL':   0.431,
        'overall_rougeL': 0.573,
        'bertscore_f1':  0.881,
        'llm_judge':     0.71,
    }
}

labels  = ['ROUGE-L\n(Overall)', 'ROUGE-L\n(Closed)', 'ROUGE-L\n(Open)', 'BERTScore F1', 'LLM-Judge']
b1_vals = [RECORDED_RESULTS['B1_zeroshot'][k] for k in ['overall_rougeL','closed_rougeL','open_rougeL','bertscore_f1','llm_judge']]
b2_vals = [RECORDED_RESULTS['B2_finetuned'][k] for k in ['overall_rougeL','closed_rougeL','open_rougeL','bertscore_f1','llm_judge']]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - width/2, b1_vals, width, label='B1 – Zero-shot', color='#5DADE2', edgecolor='white')
bars2 = ax.bar(x + width/2, b2_vals, width, label='B2 – Fine-tuned (QLoRA)', color='#2ECC71', edgecolor='white')

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('B1 Zero-shot vs B2 Fine-tuned (QLoRA) – VQA-RAD', fontsize=13, fontweight='bold')
ax.legend()
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.4, label='Threshold 0.5')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{WORKING_DIR}/b1_vs_b2_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n=== B1 vs B2 Summary ===")
for metric, b1, b2 in zip(labels, b1_vals, b2_vals):
    delta = b2 - b1
    print(f"  {metric.replace(chr(10),' '):25} B1={b1:.3f}  B2={b2:.3f}  Δ={delta:+.3f}")

In [ ]:
def is_closed(ans):
    return ans.lower().strip() in ['yes', 'no']

test_200 = test_data[:200]
closed_idx = [i for i, s in enumerate(test_200) if is_closed(s['answer'])]
open_idx = [i for i, s in enumerate(test_200) if not is_closed(s['answer'])]

closed_rL = compute_rougeL([preds_yn[i] for i in closed_idx], [gts_b1[i] for i in closed_idx])
open_rL = compute_rougeL([preds_yn[i] for i in open_idx], [gts_b1[i] for i in open_idx])

print(f"Closed-ended ROUGE-L : {closed_rL:.4f} ({len(closed_idx)} samples)")
print(f"Open-ended   ROUGE-L : {open_rL:.4f} ({len(open_idx)} samples)")
print(f"Gap (Visual Shortcut effect): {closed_rL - open_rL:+.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(['Closed-ended\n(Yes/No)', 'Open-ended\n(Giải thích)'], [closed_rL, open_rL],
              color=['#4C72B0', '#DD8452'], edgecolor='white', width=0.45)
for bar, v in zip(bars, [closed_rL, open_rL]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{v:.3f}', ha='center', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.set_ylabel('ROUGE-L F1')
ax.set_title('Visual Shortcut vs Hallucination\n(BLIP-2 Zero-shot on VQA-RAD)', fontsize=12, fontweight='bold')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Threshold 0.5')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{WORKING_DIR}/visual_shortcut_vs_collapse.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("=" * 70)
print("QUALITATIVE ANALYSIS - Zero-shot outputs")
print("=" * 70)

sample_closed = [i for i in closed_idx if i < len(preds_yn)][:4]
sample_open = [i for i in open_idx if i < len(preds_yn)][:4]

for label, indices in [('CLOSED-ENDED (Yes/No)', sample_closed), ('OPEN-ENDED (Explanation)', sample_open)]:
    print(f"\n{'-'*30} {label} {'-'*30}")
    for i in indices:
        print(f"  Q : {qs_b1[i]}")
        print(f"  GT : {gts_b1[i]}")
        print(f"  Pred (Answer) : {preds_yn[i]}")
        if preds_exp[i]:
            print(f"  Explanation   : {preds_exp[i][:150]}...")
        print()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle("BLIP-2 Zero-shot on Medical Images", fontsize=12, fontweight='bold')

sample_indices = (sample_closed + sample_open)[:6]

for ax, i in zip(axes.flatten(), sample_indices):
    s = test_200[i]
    img = s['image']
    if not isinstance(img, Image.Image):
        img = Image.fromarray(np.array(img))
    img = img.convert('RGB')
    ax.imshow(img, cmap='gray')
    ax.axis('off')
    
    match = 'V' if normalize_en(preds_yn[i]) == normalize_en(gts_b1[i]) else 'X'
    exp_short = preds_exp[i][:80] + '...' if len(preds_exp[i]) > 80 else preds_exp[i]
    title = (f"Q: {s['question'][:45]}\n"
             f"GT: {gts_b1[i]}  |  Pred: {preds_yn[i]} {match}\n"
             f"Exp: {exp_short if exp_short else '(none)'}")
    ax.set_title(title, fontsize=7.5, pad=4)

plt.tight_layout()
plt.savefig(f'{WORKING_DIR}/qualitative_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
errors = []
for i in range(min(len(preds_yn), len(test_200))):
    rL = rouge_fn.score(normalize_en(gts_b1[i]), normalize_en(preds_yn[i]))['rougeL'].fmeasure
    yn_correct = normalize_en(preds_yn[i]) == normalize_en(gts_b1[i])
    has_exp = len(preds_exp[i]) > 10
    
    if is_closed(gts_b1[i]):
        if yn_correct and has_exp:
            err_type = "Visual Shortcut (YN correct, Exp may be wrong)"
        elif not yn_correct:
            err_type = "Total Collapse (YN wrong)"
        else:
            err_type = "OK"
    else:
        if rL < 0.3:
            err_type = "Semantic Misalignment (low ROUGE-L)"
        elif rL < 0.6:
            err_type = "Partial Match"
        else:
            err_type = "OK"
    
    errors.append({
        'idx': i, 'question': qs_b1[i], 'gt': gts_b1[i],
        'pred_yn': preds_yn[i], 'rougeL': rL,
        'error_type': err_type, 'q_type': 'closed' if is_closed(gts_b1[i]) else 'open'
    })

df_err = pd.DataFrame(errors)
print(df_err['error_type'].value_counts())
print(f"\nTotal serious errors (Collapse + Misalignment): "
      f"{len(df_err[df_err['error_type'].str.contains('Collapse|Misalignment', na=False)])} / {len(df_err)}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ax.set_xlim(0, 10)
ax.set_ylim(0, 11)
ax.axis('off')
ax.set_title("BLIP-2 Zero-shot Architecture for Med-VQA", fontsize=12, fontweight='bold')

boxes = [
    (5, 10.2, "X-ray / CT / MRI", "#AED6F1", 3.5, 0.65, False),
    (5, 8.9, "ViT-L/14 Image Encoder (frozen)", "#85C1E9", 3.5, 0.65, False),
    (5, 7.5, "Q-Former Bridge\n(32 query tokens)", "#5DADE2", 3.5, 0.75, True),
    (5, 5.9, "OPT-2.7B LLM Decoder\n(8-bit, frozen)", "#2E86C1", 3.5, 0.75, True),
    (5, 4.3, "Answer: Yes/No", "#1A5276", 3.5, 0.65, True),
    (5, 3.0, "Explanation: [Analysis]", "#1A5276", 3.5, 0.65, True),
    (5, 1.6, "  Hallucination area", "#FADBD8", 3.5, 0.65, False),
]

for cx, cy, txt, col, w, h, white_text in boxes:
    rect = mpatches.FancyBboxPatch((cx - w/2, cy - h/2), w, h,
                                    boxstyle="round,pad=0.1", facecolor=col, edgecolor='white', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(cx, cy, txt, ha='center', va='center', fontsize=8.5, color='white' if white_text else 'black')

for y_top, y_bot in [(9.87, 9.22), (8.57, 7.87), (7.12, 6.27), (5.57, 4.62), (3.97, 3.32), (2.67, 1.93)]:
    ax.annotate('', xy=(5, y_bot), xytext=(5, y_top), arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))

ax.text(0.5, 0.5, "Frozen weights - no update\nUser input\nHallucination region",
        fontsize=8, va='bottom', color='#333', bbox=dict(boxstyle="round", facecolor='#f9f9f9', alpha=0.8))

plt.tight_layout()
plt.savefig(f'{WORKING_DIR}/architecture_blip2_zeroshot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import gradio as gr
from PIL import Image
import torch
import numpy as np

def predict_vqa(image, question):
    if image is None or not question.strip():
        return "Please upload an image and enter a question."
    
    try:
        if not isinstance(image, Image.Image):
            image = Image.fromarray(image)
        image = image.convert('RGB')
        
        prompt = f"Question: {question.strip()} Answer:"
        inputs = processor_b1(images=image, text=prompt, return_tensors='pt').to('cuda' if torch.cuda.is_available() else 'cpu')
        
        with torch.inference_mode():
            out = model_b1.generate(**inputs, max_new_tokens=60, num_beams=1, do_sample=False)
        
        full_text = processor_b1.decode(out[0], skip_special_tokens=True)
        if 'Answer:' in full_text:
            full_text = full_text.split('Answer:')[-1].strip()
        return full_text if full_text else "(no answer generated)"
    except Exception as e:
        return f"Error: {str(e)}"

demo = gr.Interface(
    fn=predict_vqa,
    inputs=[
        gr.Image(type="pil", label="X-ray / CT / MRI Image"),
        gr.Textbox(placeholder="E.g. Is there pleural effusion? / What is abnormal in this image?", label="Question (English)")
    ],
    outputs=gr.Textbox(label="BLIP-2 Answer (Zero-shot)"),
    title="Medical VQA – BLIP-2 Zero-shot Demo",
    description="Upload a medical image and ask a question in English. Model: Salesforce/blip2-opt-2.7b (zero-shot, not fine-tuned).",
    examples=[
        [None, "Is there pleural effusion?"],
        [None, "Where is the abnormality located?"],
        [None, "What type of imaging is this?"],
    ],
    flagging_mode="never"
)

demo.launch(share=True, debug=False)

In [ ]:
# Save predictions
df_preds = pd.DataFrame({
    'question': qs_b1[:len(preds_yn)],
    'ground_truth': gts_b1[:len(preds_yn)],
    'pred_answer': preds_yn,
    'pred_explanation': preds_exp,
})
df_preds.to_csv(f'{WORKING_DIR}/predictions_zeroshot.csv', index=False, encoding='utf-8-sig')

with open(f'{WORKING_DIR}/metrics_zeroshot.json', 'w') as f:
    json.dump(metrics_b1, f, indent=2)

print("Saved: predictions_zeroshot.csv, metrics_zeroshot.json")
print("\n=== FINAL METRICS ===")
for k, v in metrics_b1.items():
    print(f"  {k}: {v:.4f}" if v is not None else f"  {k}: N/A")

In [ ]:
!zip -r working_files.zip /kaggle/working/
from IPython.display import FileLink
FileLink(r'working_files.zip')